In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

In [ ]:
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_reference_data
%store -r restaurant_sales_data

## Static Reference Data Exploration

### Retrieval

Retrieve the four static reference data frames

In [ ]:
# Promotional items (30 rows)
before_after_details = static_reference_data['before_after_details'].copy()

# Specific customer information: for matching customers with orders
customers = static_reference_data['customers'].copy()

# Menu items for all restaurants: for matching plant-based labels with orders
items_tagged = static_reference_data['items_tagged'].copy()

# Restaurant details (30 rows)
locations = static_reference_data['locations'].copy()

# List out locations
location_ids = list(restaurant_sales_data.keys())

In [ ]:
items_tagged_new = pd.read_excel("1.5_palate_data_excel_redone/top_items_tagged.xlsx")

In [ ]:
items_tagged.drop_duplicates(subset=['item_name', 'location_id'], inplace=True)

In [ ]:
# Check if 'items_tagged' has unique pairs of 'item_name' and 'location_id'
if items_tagged.duplicated(subset=['item_name', 'location_id']).any():
    raise ValueError("Duplicates found in 'items_tagged' for the combination of 'item_name' and 'location_id'")

merged_sales_and_menu = {}
for location_id, df in tqdm(restaurant_sales_data.items()):

    # Copy to prevent overwriting
    df_copy = df.copy()
    items_tagged_copy = items_tagged.copy()

    # For ease of merging, pretend items of different capitalizations are the same <-------- *Note to potentially remove later*
    df_copy['item_name'] = df_copy['item_name'].str.lower()
    items_tagged_copy['item_name'] = items_tagged_copy['item_name'].str.lower()
    # items_tagged_copy.drop_duplicates(['item_name', 'location_id'], inplace=True) 
    
    # Perform the merge on both 'item_name' and 'location_id'
    merged = pd.merge(df_copy.reset_index(), items_tagged_copy, 
                      on=['item_name', 'location_id'], how='left')
    
    # Set 'created_at' back as the index
    merged.set_index('created_at', inplace=True)

    # Remove failed merges due to missing data on 27 and encoding errors elsewhere <-------- *Note to potentially remove later*
    merged = merged[~merged['is_plant_based'].isna()]

    # Recapitalize
    merged['item_name'].str.capitalize()

    # Store in the dictionary
    merged_sales_and_menu[location_id] = merged

### Benchmarks

Pareto analysis

In [ ]:
list_of_percentiles = []
list_of_num_dishes = []
list_of_top_items = []

j = 0
startrow = 0 
startrow2 = 0

with pd.ExcelWriter('restaurant_samples.xlsx', engine='openpyxl') as writer:

    # Determine the number of menu items for all 30 restaurants
    for loc_id in location_ids:

        header = True if startrow == 0 else False

        # Filter to the current restaurant and without alcohol
        menu_items = fdf(items_tagged).filter('location_id', loc_id).drop_duplicates('id')
        sales_and_menu_no_alcohol = fdf(merged_sales_and_menu[loc_id]).filter('dish_category', 'Alcohol', exclude=True)

        # 'entries', 'quantity', 'sales'
        target_percentage = .8
        percentiles = pa.create_percentiles(sales_and_menu_no_alcohol, metric='quantity')
        index = pa.num_items_to_cover_certain_percentage(sales_and_menu_no_alcohol, metric='quantity', percentile=target_percentage)
        
        list_of_percentiles.append(percentiles)
        list_of_num_dishes.append((index+1, percentiles.size))
        
        top_items = percentiles.iloc[:index+1]
        top_items_df = pd.DataFrame(top_items)
        top_items_df.reset_index(inplace=True)
        top_items_df['location_id'] = loc_id
        menu_lower = menu_items.copy()
        menu_lower['item_name'] = menu_lower['item_name'].str.lower()
        menu_lower.drop_duplicates(['item_name', 'location_id'], inplace=True) 
        top_items_with_id = pd.merge(top_items_df, menu_lower, on=['location_id', 'item_name'], how='inner').loc[:,['id', 'item_name', 'location_id']]
        list_of_top_items.append(top_items_with_id)

        items_tagged_copy = items_tagged.copy()
        menu_no_alcohol_all = fdf(items_tagged_copy).filter('dish_category', 'Alcohol', exclude=True)
        menu_no_alcohol = fdf(menu_no_alcohol_all).filter('location_id', loc_id)
        menu_no_alcohol['item_name'] = menu_no_alcohol['item_name'].str.lower()
        menu_no_alcohol.drop_duplicates('item_name', inplace=True)
        menu_no_alcohol['item_name'] = menu_no_alcohol['item_name'].str.capitalize()

        print(loc_id)
        is_ten = False
        n_sample = 10
        while not is_ten:
            sample = ac.calculate_accuracy(top_items, menu_no_alcohol, n_sample=n_sample).loc[:,['item_name', 'item_type', 'dish_category', 'is_plant_based', 'ingredients']]
            if sample.shape[0] == 10 or n_sample > 20:
                is_ten = True
            n_sample += 1

        pd.DataFrame(np.array([[f'Restaurant {j+1}'], [loc_id]]).T, index=['']).to_excel(writer, sheet_name='Sheet1', startrow=startrow, header=False, index=True)
        startrow += 1
    
        sample.to_excel(writer, sheet_name='Sheet1', startrow=startrow, header=header, index=True)
        top_items_with_id.to_excel(writer, sheet_name='Sheet2', startrow=startrow2, header=header, index=False)
        if header:
            startrow += 1
        startrow += sample.shape[0] + 2
        startrow2 += top_items_with_id.shape[0]
        j += 1


In [ ]:
items_tagged_new

In [ ]:
pd.concat(list_of_top_items).shape[0] + 86 - 35

In [ ]:
checking = pd.merge(items_tagged_new, pd.concat(list_of_top_items), left_on = 'item_name', right_on = 'item_name', how='outer', indicator=True)
checking[checking['_merge'] == 'left_only'].shape[0], checking[checking['_merge'] == 'right_only'].shape[0]
checking[checking['_merge'] == 'left_only']

heuristics used:
- raspberry cookies are not plantbased
- pickle chips are unclear (they said yes) (restaurant 1)
- eggplant sandwhich is not plantbased because the ingredient was given as mozzarella (opens question about ingredients)
- eggplant side without ingredient is plant based
- ginger miso listed as dumplings with no ingredients is ambigious even though they said no
- garlic fries are unclear (they said no)
- char siu was given a false positive since it was listed as unsure... (restaurant 4)
- we'll call impossible melt plant based... (restaurant 9)
- Restaurant 10 has alcohol that isnt in the alcohol section
- Restaurant 13 has a milk latte, shouldn't be plant based (noted in ingredients)
- To reiterate, things marked as unsure when they are clearly meat get a false positive



In [ ]:
sample_mislabel_list = [
{'restaurant_id':location_ids[0],
'true_positives': 3, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 1,
'ambiguous': 1},

{'restaurant_id':location_ids[1],
'true_positives': 1, 
'false_positives': 1,
'true_negatives': 4,
'false_negatives': 2,
'ambiguous': 2},

{'restaurant_id':location_ids[2],
'true_positives': 1, 
'false_positives': 2,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 5},

{'restaurant_id':location_ids[3],
'true_positives': 1, 
'false_positives': 2,
'true_negatives': 4,
'false_negatives': 1,
'ambiguous': 2},

{'restaurant_id':location_ids[4],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 0,
'ambiguous': 1},

{'restaurant_id':location_ids[5],
'true_positives': 2, 
'false_positives': 1,
'true_negatives': 4,
'false_negatives': 0,
'ambiguous': 3},

{'restaurant_id':location_ids[6],
'true_positives': 1, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 7},   # I have no idea what this stuff is

{'restaurant_id':location_ids[7],
'true_positives': 1, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 7},

{'restaurant_id':location_ids[8],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 3,
'false_negatives': 0,
'ambiguous': 3},

{'restaurant_id':location_ids[9],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 1,
'false_negatives': 5,
'ambiguous': 0},

{'restaurant_id':location_ids[10],
'true_positives': 7, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 1},

{'restaurant_id':location_ids[11],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 4},

{'restaurant_id':location_ids[12],
'true_positives': 7,    # probably more like ambigious given coffee usually has creamer
'false_positives': 3,
'true_negatives': 0,
'false_negatives': 0,
'ambiguous': 0},

{'restaurant_id':location_ids[13],
'true_positives': 2, 
'false_positives': 1,
'true_negatives': 7,
'false_negatives': 0,
'ambiguous': 0},

{'restaurant_id':location_ids[14],
'true_positives': 6, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 2},

{'restaurant_id':location_ids[15],
'true_positives': 2, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 1,
'ambiguous': 2},

{'restaurant_id':location_ids[16],
'true_positives': 3, 
'false_positives': 0,
'true_negatives': 6,
'false_negatives': 1,
'ambiguous': 0},

{'restaurant_id':location_ids[17],
'true_positives': 0, 
'false_positives': 1,
'true_negatives': 7,
'false_negatives': 0,
'ambiguous': 2},

{'restaurant_id':location_ids[18],
'true_positives': 2, 
'false_positives': 1,
'true_negatives': 4,
'false_negatives': 2,
'ambiguous': 1}, # 'Single.'

{'restaurant_id':location_ids[19],
'true_positives': 2, 
'false_positives': 3,
'true_negatives': 4,
'false_negatives': 0,
'ambiguous': 1},   # credit card fee?

{'restaurant_id':location_ids[20],
'true_positives': 3, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 5},  # "custom amount"?

{'restaurant_id':location_ids[21],
'true_positives': 1, 
'false_positives': 1,
'true_negatives': 6,
'false_negatives': 0,
'ambiguous': 2},

{'restaurant_id':location_ids[22],    # only 8 items for top 90%
'true_positives': 2, # we'll count impossible melt as yes
'false_positives': 1,
'true_negatives': 3,
'false_negatives': 0,
'ambiguous': 1},  # maybe 2 ambigious if we count gold standard kale with egg ingredient

{'restaurant_id':location_ids[23],
'true_positives': 4,  # sausage, egg, cheese (v) ? 
'false_positives': 4,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 0},

{'restaurant_id':location_ids[24],
'true_positives': 5, 
'false_positives': 1,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 2},

{'restaurant_id':location_ids[25],
'true_positives': 4, 
'false_positives': 1,
'true_negatives': 5,
'false_negatives': 0,
'ambiguous': 0},

{'restaurant_id':location_ids[26],  # only has like 10 items in top 90%
'true_positives': 2, 
'false_positives': 0,
'true_negatives': 4,
'false_negatives': 3,
'ambiguous': 1},  # "single"

{'restaurant_id':location_ids[27],   # more alcohol not labeled in alcohol category
'true_positives': 6, 
'false_positives': 0,
'true_negatives': 1,
'false_negatives': 0, 
'ambiguous': 3},   # "egift card", "foreland stolen artifacts"

{'restaurant_id':location_ids[28],
'true_positives': 3, 
'false_positives': 3,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 2},  # "card"

{'restaurant_id':location_ids[29],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 0,
'ambiguous': 1},
]

Accuracy Calculation

In [ ]:
sample_mislabel_counts = pd.DataFrame(sample_mislabel_list)

In [ ]:
sample_mislabel_counts

In [ ]:
sample_mislabel_counts = (sample_mislabel_counts
                          .assign(positive = lambda df: df['true_positives'] + df['false_negatives'],
                              negative = lambda df: df['true_negatives'] + df['false_positives'],
                              true_positive_pct = lambda df: df['true_positives'] / df['positive'],
                              false_negative_pct = lambda df: df['false_negatives'] / df['positive'],
                              true_negative_pct = lambda df: df['true_negatives'] / df['negative'],
                              false_positive_pct = lambda df: df['false_positives'] / df['negative'],
                              correct = lambda df: df['true_positives'] + df['true_negatives'],
                              incorrect = lambda df: df['false_positives'] + df['false_negatives'],
                              total = lambda df: df['correct'] + df['incorrect'],
                              correct_pct = lambda df: df['correct'] / df['total'],
                              incorrect_pct = lambda df: df['incorrect'] / df['total'])
                          #.reset_index(drop=False)
                          )

In [ ]:
# First plot: Positives
plt.figure(figsize=(10, 6))
sns.barplot(x=sample_mislabel_counts['restaurant_id'], 
            y=sample_mislabel_counts['false_negative_pct'] + sample_mislabel_counts['true_positive_pct'], 
            color="lightcoral", label="False Negatives")
sns.barplot(x=sample_mislabel_counts['restaurant_id'], 
            y=sample_mislabel_counts['true_positive_pct'], 
            color="lime", label="True Positives")
plt.title("Positive Percentages (False Negatives vs. True Positives)")
plt.xticks(rotation=90)
plt.legend()
plt.show()

# Second plot: Negatives
plt.figure(figsize=(10, 6))
sns.barplot(x=sample_mislabel_counts['restaurant_id'], 
            y=sample_mislabel_counts['false_positive_pct'] + sample_mislabel_counts['true_negative_pct'], 
            color="red", label="False Positives")
sns.barplot(x=sample_mislabel_counts['restaurant_id'], 
            y=sample_mislabel_counts['true_negative_pct'], 
            color="lightgreen", label="True Negatives")
plt.title("Negative Percentages (False Positives vs. True Negatives)")
plt.xticks(rotation=90)
plt.legend()
plt.show()

# Third plot: Correctness
plt.figure(figsize=(10, 6))
sns.barplot(x=sample_mislabel_counts['restaurant_id'], 
            y=sample_mislabel_counts['incorrect_pct'] + sample_mislabel_counts['correct_pct'], 
            color="lightcoral", label="Incorrect")
sns.barplot(x=sample_mislabel_counts['restaurant_id'], 
            y=sample_mislabel_counts['correct_pct'], 
            color="lightgreen", label="Correct")
plt.title("Correctness Percentages (Incorrect vs. Correct)")
plt.xticks(rotation=90)
plt.legend()
plt.show()

Merging the sales and menu data

In [ ]:
sample_mislabel_counts[['false_positives', 'false_negatives']].sum().sum()

In [ ]:
accuracy = sample_mislabel_counts[['true_positives', 'true_negatives']].sum().sum() / sample_mislabel_counts.sum()[1:].sum()
error_rate = sample_mislabel_counts[['false_positives', 'false_negatives']].sum().sum() / sample_mislabel_counts.sum()[1:].sum()
ambiguous = sample_mislabel_counts['ambiguous'].sum() / sample_mislabel_counts.sum()[1:].sum()
sensitivity = sample_mislabel_counts['true_positives'].sum() / sample_mislabel_counts[['true_positives', 'false_negatives']].sum().sum()
specificity = sample_mislabel_counts['true_negatives'].sum() / sample_mislabel_counts[['false_positives', 'true_negatives']].sum().sum()
precision = sample_mislabel_counts['true_positives'].sum() / sample_mislabel_counts[['true_positives', 'false_positives']].sum().sum()
negative_predictive_value = sample_mislabel_counts['true_negatives'].sum() / sample_mislabel_counts[['true_negatives', 'false_negatives']].sum().sum()

"General split:", accuracy, error_rate, ambiguous, "More details: ", sensitivity, specificity, precision, negative_predictive_value

Fixing capitalization errors in merging

In [ ]:
# Too difficult, just lower everything for counting errors

# merge_mismatches= [] 
# for loc_id in tqdm(location_ids):
#     restaurant_df = fdf(items_tagged).filter('location_id', loc_id)
#     items = restaurant_df['item_name'].tolist()
#     restaurant_merge_mismatches_list = []
#     for item in items:
#         capitalization_variation_counts = fdf(restaurant_df).filter('item_name', item)
#         restaurant_merge_mismatches_list.append((loc_id, item, capitalization_variation_counts.shape[0] - 1))
#         restaurant_merge_mismatches = pd.DataFrame(restaurant_merge_mismatches_list)
#     merge_mismatches.append((loc_id, restaurant_merge_mismatches[2].any()))

In [ ]:
list_of_new_samples = []
for loc_id in location_ids:
    df = fdf(items_tagged_new).filter('location_id', loc_id).copy()
    list_of_new_samples.append(ac.calculate_accuracy(df['item_name'].value_counts(), df))

In [ ]:
new_samples = pd.concat(list_of_new_samples)

In [ ]:
my_labels_for_new_samples_list = []
for i, row in new_samples.iterrows():
    my_label = input(row['item_name'] + ': Is it vegan?')
    my_labels_for_new_samples_list.append((row['location_id'], row['item_name'], my_label))


In [ ]:
my_labels_for_new_samples = pd.DataFrame(my_labels_for_new_samples_list, columns=['location_id','item_name','my_label'])
my_labels_for_new_samples['my_label'] = my_labels_for_new_samples['my_label'].str.capitalize()

In [ ]:
new_sample_comparison = pd.merge(new_samples, my_labels_for_new_samples, on=['location_id','item_name'], how='left').drop(columns=['id','brand'])

In [ ]:
probably_wrong = ['veg supreme', 'side slaw', 'hot chocolate']

maybe_wrong = ['chips', 'gold standard sandwich', 'americano', 'tea latte', 'milk substitute', 'kettle brand potato chips']

In [ ]:
%store new_sample_comparison